# 09 - Final Merge Test for RS-PPO / ArmoRM (Deliberately Circular)

This notebook runs the binding primary endpoint after Notebook 08 has produced the PPO adapters, geometry matrices, reward matrix over B, Wall-A diagnostics, and provenance sidecars. It is still deliberately circular: PPO was trained against ArmoRM and this notebook evaluates against ArmoRM, so it must not be reported as RQ2 proxy validation.

The valid claim is the upper-bound test: if an exact effective-LoRA merge cannot make `f(p,R)` beat `lambda=p` here, endpoint-linear coefficient correction is capped even in the best-case regime.


In [ ]:
%cd /content

import os, sys, json, math, random, shutil, subprocess, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

repo_path = Path('/content/master-thesis')
repo_url = 'https://github.com/NZhang137/master-thesis.git'
if (repo_path / '.git').is_dir():
    print('Repository exists; pulling latest changes.')
    subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only'], check=False)
else:
    print('Repository missing; cloning from GitHub.')
    if repo_path.exists():
        shutil.rmtree(repo_path)
    subprocess.run(['git', 'clone', repo_url, str(repo_path)], check=True)

%cd /content/master-thesis

!pip install -q "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" "trl==0.8.6" bitsandbytes datasets scipy numpy pandas matplotlib safetensors

import importlib
import numpy as np
import pandas as pd
import torch

CONFIG_09 = {
    'SEED': 137,
    'OUTPUT_DIR': 'results/rs_ppo_armorm_circular',
    'OUTPUT_ZIP': 'rs_ppo_final_merge_outputs.zip',
    'BASE_MODEL': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'ARMORM_MODEL': 'RLHFlow/ArmoRM-Llama3-8B-v0.1',
    'ATTRIBUTES': ['helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity'],
    'CIRCULAR_ARMORM_ACKNOWLEDGED': True,
    'M1PLUS_RHO': 0.5,
    'C1PP_C': 0.5,
    'C1PP_EPS': 1e-8,
    'BOOTSTRAP_N': 2000,
    'BOOTSTRAP_SEED': 137,
    'REWARD_NUM_PROMPTS': 80,
    'REWARD_PROMPT_SPLIT': 'validation',
    'REWARD_PROMPT_OFFSET': 160,
    'N_GEN_PER_PROMPT': 4,
    'MERGE_DTYPE': 'float32',
    'MIN_GPU_MEMORY_GB': 35.0,
    'DM_DENOM_MIN': 1e-3,
    'HOLM_ALPHA': 0.05,
    'RUN_MERGE': False,
}

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = (PROJECT_ROOT / CONFIG_09['OUTPUT_DIR']).resolve()
RS_RUNS_DIR = OUTPUT_DIR / 'rs_runs'
SFT_MERGED = RS_RUNS_DIR / 'theta_sft' / 'merged'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG_09['SEED']); np.random.seed(CONFIG_09['SEED']); torch.manual_seed(CONFIG_09['SEED'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG_09['SEED'])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.helpsteer2_utils import HELPSTEER2_ATTRIBUTES
assert tuple(CONFIG_09['ATTRIBUTES']) == tuple(HELPSTEER2_ATTRIBUTES), (
    f"axis order drift: {CONFIG_09['ATTRIBUTES']} vs {HELPSTEER2_ATTRIBUTES}")

rs_ppo = importlib.import_module('scripts.train_rs_ppo')
rs_ppo = importlib.reload(rs_ppo)
blocked = False
try:
    rs_ppo.check_reward_firewall('helpfulness', CONFIG_09['ARMORM_MODEL'],
                                 circular_armorm_acknowledged=False)
except AssertionError as error:
    blocked = True
    print('Firewall correctly blocks unacknowledged ArmoRM PPO:', error)
assert blocked, 'FIREWALL BROKEN: unacknowledged ArmoRM PPO was allowed.'
firewall_ack = rs_ppo.check_reward_firewall('helpfulness', CONFIG_09['ARMORM_MODEL'],
                                            circular_armorm_acknowledged=True)
assert firewall_ack['circularity_acknowledged'] is True
assert 'RQ2 (proxy validity)' in firewall_ack['retired_research_questions']

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n', encoding='utf-8')
    assert path.exists(), f'Missing JSON output: {path}'

def write_numpy(path, values):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, np.asarray(values))
    assert path.exists(), f'Missing NumPy output: {path}'

print('Project root:', PROJECT_ROOT)
print('Output dir:', OUTPUT_DIR)
print(json.dumps(CONFIG_09, indent=2, sort_keys=True))


## Input Integrity Gate

This cell checks all Notebook 08 artifacts and merge provenance before any GPU scoring is allowed. If the reward matrix was produced by the old PEFT `linear` merge, this notebook aborts.


In [ ]:
R_COS_PATH = OUTPUT_DIR / 'R_cos.npy'
REWARD_MATRIX_PATH = OUTPUT_DIR / 'reward_matrix.npy'
REWARD_MATRIX_META_PATH = OUTPUT_DIR / 'reward_matrix_meta.json'
REWARD_PROMPTS_PATH = OUTPUT_DIR / 'reward_prompts.json'
SEARCH_SET_PATH = OUTPUT_DIR / 'search_set_B.npy'
PREREGISTRATION_PATH = OUTPUT_DIR / 'preregistration.json'
PLATEAU_PATH = OUTPUT_DIR / 'plateau_report.json'
LINEARITY_R2_PATH = OUTPUT_DIR / 'linearity_r2.json'
LAMBDA_STAR_PATH = OUTPUT_DIR / 'lambda_star.json'
HEADS_BY_LAMBDA_PATH = OUTPUT_DIR / 'heads_by_lambda.npz'
SCORING_STATUS_PATH = OUTPUT_DIR / 'merge_scoring_status.json'
PREREG_ADDENDUM_PATH = OUTPUT_DIR / 'preregistration_addendum_nb09.json'
MERGE_RESULTS_PATH = OUTPUT_DIR / 'merge_results.json'

required_paths = [
    R_COS_PATH, REWARD_MATRIX_PATH, REWARD_MATRIX_META_PATH, REWARD_PROMPTS_PATH,
    SEARCH_SET_PATH, PREREGISTRATION_PATH, PLATEAU_PATH, LINEARITY_R2_PATH, SFT_MERGED,
]
required_paths += [RS_RUNS_DIR / f'ppo_{axis}' / 'adapter' for axis in CONFIG_09['ATTRIBUTES']]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, 'Missing required Notebook 08 artifacts: ' + ', '.join(missing)

reward_meta = json.loads(REWARD_MATRIX_META_PATH.read_text(encoding='utf-8'))
assert reward_meta.get('merge_impl') == 'src.merge.merge_theta', (
    "reward_matrix.npy was produced with the old peft 'linear' merge (cross-term contaminated). "
    'Re-run Notebook 08 reward collection with src.merge before using it.')
VERIFIER_REPORT_PATH = PROJECT_ROOT / 'results' / 'verify_rs_ppo_setup' / 'report.json'
assert VERIFIER_REPORT_PATH.exists(), (
    f'{VERIFIER_REPORT_PATH} missing. Run scripts/verify_rs_ppo_setup.py before the binding run.')
verifier_report = json.loads(VERIFIER_REPORT_PATH.read_text(encoding='utf-8'))
verifier_status = {row['check_id']: row['status'] for row in verifier_report.get('results', [])}
assert verifier_status.get('G7') == 'PASS', (
    'G7 (production merge vs independent safetensors oracle) is not PASS in report.json. '
    'The merge path is unverified; no GPU time may be spent.')
for gpu_check in ('M4', 'M5'):
    if gpu_check in verifier_status and verifier_status[gpu_check] != 'PASS':
        raise AssertionError(f'{gpu_check} did not PASS in report.json: {verifier_status[gpu_check]}')
print('Verifier report:', {k: verifier_status[k] for k in sorted(verifier_status)})

# The provenance flag must have been DERIVED from the verifier report by NB08, not asserted
# by NB08 about itself. We check the derivation is present and consistent with our own report.
assert isinstance(reward_meta.get('merge_verification'), dict), (
    "reward_matrix_meta.json has no 'merge_verification' block; it was written by an older, "
    'self-certifying version of Notebook 08. Re-run the reward collection.')
assert 'G7' in reward_meta['merge_verification'].get('checks_passed', []), reward_meta['merge_verification']
assert reward_meta.get('merge_linearity_verified') is True, (
    "reward_matrix.npy was produced with the old peft 'linear' merge (cross-term contaminated). "
    'Re-run Notebook 08 reward collection with src.merge before using it.')
assert int(reward_meta.get('prompt_offset')) == CONFIG_09['REWARD_PROMPT_OFFSET'], reward_meta
assert 'git_sha' in reward_meta and reward_meta['git_sha'], reward_meta
assert int(reward_meta.get('n_gen_per_prompt')) == CONFIG_09['N_GEN_PER_PROMPT'], (
    'reward_matrix.npy was built with a different n_gen_per_prompt. '
    'Re-run Notebook 08 reward collection with the current CONFIG.', reward_meta)
assert reward_meta.get('merge_dtype') == CONFIG_09['MERGE_DTYPE'], (
    'reward_matrix.npy was built with a different merge dtype. '
    'Re-run Notebook 08 reward collection with the current CONFIG.', reward_meta)
try:
    current_git_sha = subprocess.check_output(
        ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
except Exception:
    current_git_sha = 'unknown'
if reward_meta.get('git_sha') != current_git_sha:
    print('*** WARNING: reward_matrix git_sha differs from current HEAD:',
          reward_meta.get('git_sha'), '!=', current_git_sha)

plateau_report = json.loads(PLATEAU_PATH.read_text(encoding='utf-8'))
linearity_r2 = json.loads(LINEARITY_R2_PATH.read_text(encoding='utf-8'))
print('Reward matrix provenance:', json.dumps(reward_meta, indent=2, sort_keys=True))
print('Plateau summary:', json.dumps(plateau_report.get('_summary', plateau_report), indent=2, sort_keys=True))
print('Wall-A summary:', json.dumps({k: linearity_r2.get(k) for k in ['wall_a_stands', 'quality_r2_max', 'wall_A_R2_threshold']}, indent=2))
if plateau_report.get('_summary', {}).get('upper_bound_reading_valid') is False:
    print('*** WARNING: upper-bound reading invalid for plateaued axes:',
          plateau_report['_summary'].get('axes_plateaued'))


## Pre-Registration Addendum

This addendum freezes the merge-fix provenance and the Holm-corrected family decision rule before this notebook contacts ArmoRM.


In [ ]:
nb08_prereg = json.loads(PREREGISTRATION_PATH.read_text(encoding='utf-8'))
PRIMARY_METRIC = nb08_prereg['primary']['metric']
metric_upper = PRIMARY_METRIC.upper()
assert 'PAIRED' in metric_upper and 'PER-PROMPT' in metric_upper
assert ('RAW ARMORM' in metric_upper) or ('NATIVE ARMORM' in metric_upper)
assert nb08_prereg.get('superseded_by') == 'preregistration_addendum_nb09.json (Holm-corrected, per-family)'

precision_report = {'pending': True, 'source': 'scripts/verify_rs_ppo_setup.py::M5'}
verify_report_path = PROJECT_ROOT / 'results' / 'verify_rs_ppo_setup' / 'report.json'
if verify_report_path.exists():
    verify_report = json.loads(verify_report_path.read_text(encoding='utf-8'))
    for result in verify_report.get('results', []):
        if result.get('check_id') == 'M5' and result.get('status') == 'PASS':
            precision_report = {'pending': False, **result.get('details', {})}

addendum = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'confirmatory': True,
    'circular_do_not_report_as_proxy_validation': True,
    'primary': {
        'metric': PRIMARY_METRIC,
        'success_rule': (
            'For each moving preference, bootstrap Delta U_p over the same 80 prompt indices. '
            'Within each regime family, Holm-adjust p-values. A preference succeeds iff '
            'Holm-adjusted p < HOLM_ALPHA and mean Delta U_p > 0. The quality-regime verdict '
            'is positive iff at least one moving quality preference succeeds and no associated axis plateaued.'),
        'holm_alpha': CONFIG_09['HOLM_ALPHA'],
        'bootstrap_n': CONFIG_09['BOOTSTRAP_N'],
        'bootstrap_seed': CONFIG_09['BOOTSTRAP_SEED'],
    },
    'pre_run_tightenings': {
        'multiplicity': {
            'method': 'Holm correction within moving preferences of each regime family',
            'families': {'quality': 6, 'complexity_verbosity': 4, 'uniform': 'reported separately'},
            'no_movement_rule': (
                'Preferences with lambda* == p are excluded from Holm family size and sign-test denominator; '
                'they are reported as not testable because Delta U_p is identically zero by portfolio certificate.'),
            'rationale': (
                'A single significant quality preference would overturn the upper-bound negative result. '
                'Six uncorrected tests would inflate the family-wise false-positive rate, so correction is '
                'required exactly for the positive claim.')
        },
        'regime_level_aggregate': {
            'method': 'block bootstrap over prompt indices with one shared resample matrix reused across preferences'
        },
        'merge_fix': {
            'merge_impl': 'src.merge.merge_theta',
            'previous_path': "PEFT add_weighted_adapter(combination_type='linear')",
            'previous_path_problem': 'cross-term contaminated interior lambdas',
        },
        'merge_dtype': {
            'from': 'bf16 storage',
            'to': CONFIG_09['MERGE_DTYPE'],
            'reason': 'small theta(lambda*) - theta(p) displacements can be swamped by bf16 rounding',
            'm5_precision_check': precision_report,
        },
        'delta_m_percent': {
            'reported_secondary_metric': 'delta_m_percent_unweighted_gain',
            'context_metric': 'delta_m_percent_pref_weighted_gain',
            'rationale': 'unweighted Delta m% is the MTL field-standard mean over heads; preference-weighted is context only.'
        }
    },
    'artifacts_read': {
        'nb08_preregistration': str(PREREGISTRATION_PATH),
        'reward_matrix_meta': str(REWARD_MATRIX_META_PATH),
    },
}
write_json(PREREG_ADDENDUM_PATH, addendum)
print(json.dumps(addendum, indent=2, sort_keys=True))


## Coefficients and Floor Certificates

This CPU-only cell computes `lambda*_M1+`, portfolio certificates, p-aware floor LP values, and the deduplicated lambda set before any GPU time is spent.


In [ ]:
from src.coefficient_portfolio import floor_lp_at_p, run_portfolio
from src.lambda_utils import lambda_key
from src.preferences import CV_PREFS, PREFERENCES, QUALITY_PREFS, preference_regime

R = np.load(R_COS_PATH)
assert R.shape == (5, 5)
unique_lambdas = {}
rows = []
for name, p_list in PREFERENCES.items():
    p_vec = np.asarray(p_list, dtype=np.float64)
    assert np.all(p_vec >= 0) and np.isclose(p_vec.sum(), 1.0), name
    portfolio = run_portfolio(p_vec, R, CONFIG_09)
    lam = np.asarray(portfolio['M1+']['lam'], dtype=np.float64)
    assert np.all(np.isfinite(lam)) and np.all(lam >= -1e-8) and np.isclose(lam.sum(), 1.0), name
    for method in ('M1+', 'C1++', 'M1++', 'P2++', 'P3++'):
        assert method in portfolio and 'returns_p' in portfolio[method], f'missing portfolio result: {method}'
    floor_value, floor_collapsed = floor_lp_at_p(R, p_vec, tol=1e-9)
    no_movement = bool(np.linalg.norm(lam - p_vec) < 1e-6)
    unique_lambdas[lambda_key(p_vec)] = p_vec.tolist()
    if not no_movement:
        unique_lambdas[lambda_key(lam)] = lam.tolist()
    rows.append({
        'preference': name,
        'regime': preference_regime(name),
        'p': p_vec.tolist(),
        'lambda_star_M1plus': lam.tolist(),
        'l2_lambda_minus_p': float(np.linalg.norm(lam - p_vec)),
        'no_movement': no_movement,
        'portfolio': portfolio,
        'floor_lp_at_p': {'value': float(floor_value), 'collapsed': bool(floor_collapsed)},
    })

lambda_payload = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preferences': rows,
    'unique_lambdas': unique_lambdas,
    'n_unique_lambdas_to_score': len(unique_lambdas),
    'gpu_cost_estimate_generations': len(unique_lambdas) * CONFIG_09['REWARD_NUM_PROMPTS'] * CONFIG_09['N_GEN_PER_PROMPT'],
    'config': CONFIG_09,
}
write_json(LAMBDA_STAR_PATH, lambda_payload)
print('Unique lambdas to score:', len(unique_lambdas))
print('GPU cost estimate generations:', lambda_payload['gpu_cost_estimate_generations'])
display(pd.DataFrame(rows)[['preference', 'regime', 'l2_lambda_minus_p', 'no_movement', 'floor_lp_at_p']])


## GPU Scoring

This expensive cell is behind `RUN_MERGE`. It scores each unique lambda exactly once, caches raw per-prompt head tensors, and aborts on degenerate reward-head outputs.


In [ ]:
if CONFIG_09['RUN_MERGE']:
    assert torch.cuda.is_available(), 'RUN_MERGE requires CUDA.'
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    assert gpu_gb >= CONFIG_09['MIN_GPU_MEMORY_GB'], (
        f"TinyLlama fp32 merge + ArmoRM 4-bit requires an A100/high-memory runtime; got {gpu_gb:.1f} GB")
    from datasets import load_dataset
    from transformers import AutoTokenizer
    from src.merge import merge_theta

    lambda_payload = json.loads(LAMBDA_STAR_PATH.read_text(encoding='utf-8'))
    unique_lambdas = {k: np.asarray(v, dtype=np.float64) for k, v in lambda_payload['unique_lambdas'].items()}
    adapter_paths = {axis: RS_RUNS_DIR / f'ppo_{axis}' / 'adapter' for axis in CONFIG_09['ATTRIBUTES']}
    assert all(path.is_dir() for path in adapter_paths.values())
    assert SFT_MERGED.exists()

    saved_prompts = json.loads(REWARD_PROMPTS_PATH.read_text(encoding='utf-8'))['prompts']
    ds = load_dataset('nvidia/HelpSteer2', split=CONFIG_09['REWARD_PROMPT_SPLIT'])
    off = CONFIG_09['REWARD_PROMPT_OFFSET']
    eval_prompts = [str(ds[i]['prompt']) for i in range(off, off + CONFIG_09['REWARD_NUM_PROMPTS'])]
    assert eval_prompts == saved_prompts, 'NB09 prompts are not byte-identical to NB08 reward_prompts.json'

    scorer = rs_ppo.ArmoRMHeadScorer(axis='helpfulness', model_id=CONFIG_09['ARMORM_MODEL'])
    scorer.validate_batching([(q, 'probe response') for q in eval_prompts[:8]])
    tok = AutoTokenizer.from_pretrained(str(SFT_MERGED))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    length_rng = np.random.default_rng(CONFIG_09['SEED'])
    generation_lengths = [int(length_rng.integers(rs_ppo.CFG['output_min_len'],
                                                  rs_ppo.CFG['output_max_len'] + 1))
                          for _ in range(CONFIG_09['N_GEN_PER_PROMPT'])]

    def score_merged_lambda(lmbda):
        rs_ppo.set_all_seeds(CONFIG_09['SEED'])
        model = merge_theta(lmbda, adapter_paths, SFT_MERGED, dtype=CONFIG_09['MERGE_DTYPE'])
        repeat_heads = []
        for repeat, n_new in enumerate(generation_lengths):
            responses = []
            for start in range(0, len(eval_prompts), 8):
                chunk = eval_prompts[start:start + 8]
                texts = [tok.apply_chat_template([{'role': 'user', 'content': q}],
                                                 tokenize=False, add_generation_prompt=True)
                         for q in chunk]
                enc = tok(texts, return_tensors='pt', padding=True).to(model.device)
                with torch.inference_mode():
                    out = model.generate(**enc, max_new_tokens=n_new, do_sample=True,
                                         top_k=0, top_p=1.0, pad_token_id=tok.eos_token_id)
                responses.extend(tok.batch_decode(out[:, enc['input_ids'].shape[1]:],
                                                  skip_special_tokens=True))
            repeat_heads.append(scorer.score_all_heads(eval_prompts, responses))
        heads = np.mean(np.stack(repeat_heads, axis=0), axis=0).astype(np.float32)
        del model
        torch.cuda.empty_cache()
        return heads

    heads_cache = {}
    t0 = time.time()
    for idx_key, (key, lmbda) in enumerate(unique_lambdas.items(), start=1):
        heads = score_merged_lambda(lmbda)
        stds = heads.std(axis=0)
        uniques = np.asarray([len(np.unique(heads[:, i])) for i in range(heads.shape[1])])
        failures = [f'{axis}: std={stds[i]:.3e}, unique={uniques[i]}'
                    for i, axis in enumerate(CONFIG_09['ATTRIBUTES'])
                    if stds[i] <= 1e-6 or uniques[i] <= 10]
        assert not failures, 'Degenerate ArmoRM head outputs: ' + '; '.join(failures)
        heads_cache[key] = heads
        elapsed = time.time() - t0
        eta = elapsed / idx_key * (len(unique_lambdas) - idx_key)
        print(f'lambda {idx_key}/{len(unique_lambdas)} elapsed={elapsed/60:.1f} min eta={eta/60:.1f} min')

    np.savez_compressed(HEADS_BY_LAMBDA_PATH, **heads_cache)
    write_json(SCORING_STATUS_PATH, {
        'pending': False,
        'heads_by_lambda': str(HEADS_BY_LAMBDA_PATH),
        'n_unique_lambdas': len(unique_lambdas),
        'generation_lengths': generation_lengths,
        'n_gen_per_prompt': CONFIG_09['N_GEN_PER_PROMPT'],
        'merge_dtype': CONFIG_09['MERGE_DTYPE'],
    })
    del scorer
    torch.cuda.empty_cache()
else:
    write_json(SCORING_STATUS_PATH, {'pending': True, 'reason': 'RUN_MERGE is False; no fake GPU scores.'})
    print('RUN_MERGE is False; GPU scoring skipped and no fake numbers were created.')


## Analysis

This CPU-only cell reads `heads_by_lambda.npz` and can be rerun without GPU work. It computes the raw paired primary endpoint, paired rank secondary endpoint, Holm correction, regime block bootstraps, and the final upper-bound verdict.


In [ ]:
from scipy.stats import binomtest
from src.coefficient_portfolio import paired_rank_delta
from src.lambda_utils import holm_adjust, lambda_key
from src.metrics import delta_m_percent
from src.preferences import CV_PREFS, PREFERENCES, QUALITY_PREFS, preference_regime

if HEADS_BY_LAMBDA_PATH.exists():
    scoring_status = json.loads(SCORING_STATUS_PATH.read_text(encoding='utf-8'))
    assert scoring_status.get('pending') is False, 'heads_by_lambda.npz exists but scoring status is pending/stale.'
    assert int(scoring_status.get('n_gen_per_prompt')) == CONFIG_09['N_GEN_PER_PROMPT'], scoring_status
    assert scoring_status.get('merge_dtype') == CONFIG_09['MERGE_DTYPE'], scoring_status
    lambda_payload = json.loads(LAMBDA_STAR_PATH.read_text(encoding='utf-8'))
    heads_npz = np.load(HEADS_BY_LAMBDA_PATH)
    Reward = np.load(REWARD_MATRIX_PATH)
    stl = np.diag(Reward[:5]).astype(float)
    assert np.all(np.abs(stl) > CONFIG_09['DM_DENOM_MIN']), f'Delta m% denominator too small: {stl}'
    plateau_report = json.loads(PLATEAU_PATH.read_text(encoding='utf-8'))
    linearity_r2 = json.loads(LINEARITY_R2_PATH.read_text(encoding='utf-8'))

    rng = np.random.default_rng(CONFIG_09['BOOTSTRAP_SEED'])
    idx = rng.integers(0, CONFIG_09['REWARD_NUM_PROMPTS'],
                       (CONFIG_09['BOOTSTRAP_N'], CONFIG_09['REWARD_NUM_PROMPTS']))

    rows = []
    per_prompt_by_pref = {}
    primary_axis_by_pref = {
        'dominant_helpfulness': 'helpfulness', 'only_helpfulness': 'helpfulness',
        'dominant_correctness': 'correctness', 'only_correctness': 'correctness',
        'dominant_coherence': 'coherence', 'only_coherence': 'coherence',
        'dominant_complexity': 'complexity', 'only_complexity': 'complexity',
        'dominant_verbosity': 'verbosity', 'only_verbosity': 'verbosity',
        'uniform': None,
    }

    for row in lambda_payload['preferences']:
        name = row['preference']
        p_vec = np.asarray(row['p'], dtype=np.float64)
        lam = np.asarray(row['lambda_star_M1plus'], dtype=np.float64)
        heads_p = heads_npz[lambda_key(p_vec)]
        heads_l = heads_p if row['no_movement'] else heads_npz[lambda_key(lam)]
        per_prompt = (heads_l - heads_p) @ p_vec
        per_prompt_by_pref[name] = per_prompt
        boots = per_prompt[idx].mean(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        boot_p = float(min(1.0, 2.0 * min(np.mean(boots <= 0.0), np.mean(boots >= 0.0))))

        per_prompt_rank = paired_rank_delta(heads_p, heads_l, p_vec)
        boots_r = per_prompt_rank[idx].mean(axis=1)
        lo_r, hi_r = np.percentile(boots_r, [2.5, 97.5])

        dm_p_unweighted = delta_m_percent(heads_p.mean(0), stl, p=None, denom_min=CONFIG_09['DM_DENOM_MIN'])
        dm_l_unweighted = delta_m_percent(heads_l.mean(0), stl, p=None, denom_min=CONFIG_09['DM_DENOM_MIN'])
        dm_p_weighted = delta_m_percent(heads_p.mean(0), stl, p=p_vec, denom_min=CONFIG_09['DM_DENOM_MIN'])
        dm_l_weighted = delta_m_percent(heads_l.mean(0), stl, p=p_vec, denom_min=CONFIG_09['DM_DENOM_MIN'])

        rows.append({
            'preference': name,
            'regime': row['regime'],
            'p': row['p'],
            'lambda_star_M1plus': row['lambda_star_M1plus'],
            'l2_lambda_minus_p': row['l2_lambda_minus_p'],
            'no_movement': row['no_movement'],
            'testable': not row['no_movement'],
            'not_testable_reason': ('lambda* == p; Delta U_p is identically zero by portfolio certificate'
                                    if row['no_movement'] else None),
            'delta_U_p_mean': float(per_prompt.mean()),
            'delta_U_p_ci95': [float(lo), float(hi)],
            'boot_p': boot_p,
            'holm_p': None,
            'significant': False,
            'delta_U_p_rank_mean': float(per_prompt_rank.mean()),
            'delta_U_p_rank_ci95': [float(lo_r), float(hi_r)],
            'delta_m_percent_unweighted_gain': dm_l_unweighted - dm_p_unweighted,
            'delta_m_percent_pref_weighted_gain': dm_l_weighted - dm_p_weighted,
            'portfolio': row['portfolio'],
            'floor_lp_at_p': row['floor_lp_at_p'],
            'primary_axis': primary_axis_by_pref[name],
        })

    for family in (QUALITY_PREFS, CV_PREFS):
        indices = [i for i, row in enumerate(rows)
                   if row['preference'] in family and row['testable']]
        adjusted = holm_adjust([rows[i]['boot_p'] for i in indices]) if indices else []
        for local_idx, global_idx in enumerate(indices):
            rows[global_idx]['holm_p'] = float(adjusted[local_idx])
            rows[global_idx]['significant'] = bool(
                adjusted[local_idx] < CONFIG_09['HOLM_ALPHA']
                and rows[global_idx]['delta_U_p_mean'] > 0
            )

    regime_aggregates = {}
    for label, names in {'quality': QUALITY_PREFS, 'cv': CV_PREFS}.items():
        testable_names = [name for name in names
                          if any(row['preference'] == name and row['testable'] for row in rows)]
        if testable_names:
            arr = np.stack([per_prompt_by_pref[name] for name in testable_names], axis=0)
            boots = arr[:, idx].mean(axis=2).mean(axis=0)
            lo, hi = np.percentile(boots, [2.5, 97.5])
            regime_aggregates[label] = {'mean': float(arr.mean()), 'ci95': [float(lo), float(hi)],
                                        'n_testable': len(testable_names)}
        else:
            regime_aggregates[label] = {'pending': True, 'reason': 'no moving preferences in family',
                                        'n_testable': 0}

    moving_rows = [row for row in rows if row['testable']]
    n_pos = int(sum(row['delta_U_p_mean'] > 0 for row in moving_rows))
    sign_p = (float(binomtest(n_pos, len(moving_rows), 0.5, alternative='greater').pvalue)
              if moving_rows else None)
    n_quality_significant_after_holm = int(sum(row['significant'] for row in rows if row['preference'] in QUALITY_PREFS))
    n_cv_significant_after_holm = int(sum(row['significant'] for row in rows if row['preference'] in CV_PREFS))

    plateau_axes = set(plateau_report.get('_summary', {}).get('axes_plateaued', []))
    significant_quality = [row for row in rows if row['preference'] in QUALITY_PREFS and row['significant']]
    invalid_axes = sorted({row['primary_axis'] for row in significant_quality if row['primary_axis'] in plateau_axes})
    if n_quality_significant_after_holm > 0:
        upper_bound_verdict = 'Wall A was not a hard cap; strong positive result.'
        if invalid_axes:
            upper_bound_verdict += ' However, upper_bound_argument_invalid_for_this_axis=' + ','.join(invalid_axes)
    else:
        upper_bound_verdict = (
            'UPPER-BOUND NEGATIVE RESULT: endpoint-linear coefficient correction cannot trace '
            'a nonlinear reward landscape, even in the best-case circular regime.')

    result_payload = {
        'circular_do_not_report_as_proxy_validation': True,
        'primary_endpoint': json.loads(PREREG_ADDENDUM_PATH.read_text(encoding='utf-8'))['primary']['metric'],
        'reported_secondary_metric': 'delta_m_percent_unweighted_gain',
        'context_secondary_metric': 'delta_m_percent_pref_weighted_gain',
        'holm_alpha': CONFIG_09['HOLM_ALPHA'],
        'holm_family_rule': 'moving preferences only; no_movement rows are not testable',
        'plateau_status_per_axis': plateau_report,
        'wall_a_stands': linearity_r2.get('wall_a_stands'),
        'rows': rows,
        'regime_aggregates': regime_aggregates,
        'n_positive_moving': n_pos,
        'n_moving_preferences': len(moving_rows),
        'sign_test_p_one_sided': sign_p,
        'n_quality_significant_after_holm': n_quality_significant_after_holm,
        'n_cv_significant_after_holm': n_cv_significant_after_holm,
        'upper_bound_argument_invalid_for_axes': invalid_axes,
        'upper_bound_verdict': upper_bound_verdict,
        'quality_regime_positive': bool(n_quality_significant_after_holm > 0 and not invalid_axes),
    }
    write_json(MERGE_RESULTS_PATH, result_payload)
    display(pd.DataFrame(rows)[['preference', 'regime', 'testable', 'delta_U_p_mean', 'delta_U_p_ci95',
                                'boot_p', 'holm_p', 'significant',
                                'delta_m_percent_unweighted_gain',
                                'delta_m_percent_pref_weighted_gain']])
    print('Upper-bound verdict:', upper_bound_verdict)
else:
    write_json(MERGE_RESULTS_PATH, {
        'pending': True,
        'reason': 'heads_by_lambda.npz missing; run the GPU scoring cell with RUN_MERGE=True.',
    })
    print('Analysis pending: heads_by_lambda.npz missing.')


## Zip and Verdict

Packages Notebook 09 artifacts and prints a compact STOP/GO summary.


In [ ]:
ZIP_PATH = OUTPUT_DIR / CONFIG_09['OUTPUT_ZIP']
SUMMARY_09_PATH = OUTPUT_DIR / 'summary_nb09.md'
merge_results = json.loads(MERGE_RESULTS_PATH.read_text(encoding='utf-8')) if MERGE_RESULTS_PATH.exists() else {'pending': True}

summary = ['# 09 - Final Merge Test', '']
summary += ['- circular_do_not_report_as_proxy_validation: true']
summary += [f"- pending: {merge_results.get('pending', False)}"]
summary += [f"- quality_regime_positive: {merge_results.get('quality_regime_positive')}"]
summary += [f"- run_completed: {not merge_results.get('pending', False)}"]
summary += [f"- upper_bound_verdict: {merge_results.get('upper_bound_verdict')}"]
SUMMARY_09_PATH.write_text('\n'.join(summary) + '\n', encoding='utf-8')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [LAMBDA_STAR_PATH, HEADS_BY_LAMBDA_PATH, MERGE_RESULTS_PATH,
                 PREREG_ADDENDUM_PATH, SCORING_STATUS_PATH, SUMMARY_09_PATH]:
        path = Path(path)
        if path.exists():
            archive.write(path, arcname=path.name)
            print('Added:', path.name)

print('Output zip:', ZIP_PATH)
print('Run completed:', not merge_results.get('pending', False))
print('Verdict:', merge_results.get('upper_bound_verdict', merge_results.get('reason')))
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as error:
    print('Download manually from:', ZIP_PATH, '|', repr(error))
